# Maths Required for Machine Learning

Every ML algorithm is built on a small set of mathematical ideas. You don't need to be a mathematician — but understanding these concepts will make algorithms feel logical rather than magical.

## Topics Covered

| # | Topic | Why it matters in ML |
|---|---|---|
| 1 | **Basic Statistics** | Understand your data — mean, variance, spread |
| 2 | **Probability** | Models output probabilities; understand what they mean |
| 3 | **Linear Algebra** | Features are vectors; model parameters are matrices |
| 4 | **Derivatives & Gradient** | How models learn — gradient descent |
| 5 | **Distance Metrics** | KNN, K-Means, SVM all measure "closeness" |
| 6 | **Error & Cost Functions** | What the model is trying to minimise during training |

## 1 — Basic Statistics

Before fitting any model, you need to understand the numbers in your dataset.

| Measure | Formula | What it tells you |
|---|---|---|
| **Mean** | Σx / n | Central value (average) |
| **Median** | Middle value when sorted | Centre that is robust to outliers |
| **Mode** | Most frequent value | Most common category |
| **Variance** | Σ(x − μ)² / n | How spread out the values are |
| **Std Deviation** | √Variance | Spread in same unit as the data |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# Student exam scores
scores = np.array([55, 62, 70, 72, 74, 75, 76, 78, 80, 82, 85, 88, 90, 92, 95, 45, 98, 60, 73, 77])

# ── MEAN ─────────────────────────────────────────────────────────────────────
mean = scores.mean()  # = sum / count
print(f'Mean   : {mean:.2f}   (sum={scores.sum()} ÷ n={len(scores)})')

# ── MEDIAN ───────────────────────────────────────────────────────────────────
median = np.median(scores)
# Middle value of sorted data → not affected by outliers
# If mean >> median → right-skewed (a few very high values pulling mean up)
print(f'Median : {median:.2f}   (middle of sorted data)')
print(f'  Mean ({mean:.1f}) vs Median ({median:.1f}) → difference of {abs(mean-median):.1f} suggests mild skew')

# ── MODE ─────────────────────────────────────────────────────────────────────
count = Counter(scores)
mode = count.most_common(1)[0][0]
print(f'Mode   : {mode}     (most frequent value, count={count[mode]})')

# ── VARIANCE ─────────────────────────────────────────────────────────────────
variance = scores.var()  # = mean of (x - mean)²
# High variance = scores are spread far from the mean
# Low variance  = scores are clustered tightly around the mean
print(f'\nVariance  : {variance:.2f}   (mean of squared deviations from mean)')

# ── STANDARD DEVIATION ────────────────────────────────────────────────────────
std = scores.std()  # = √variance — same unit as the original data (marks)
print(f'Std Dev   : {std:.2f}   (√variance — in same units as scores)')
print(f'\n  Interpretation: most scores fall within mean ± std')
print(f'  → [{mean:.1f} - {std:.1f}, {mean:.1f} + {std:.1f}] = [{mean-std:.1f}, {mean+std:.1f}]')
print(f'  → {((scores >= mean-std) & (scores <= mean+std)).sum()}/{len(scores)} scores are in this range')

# ── PERCENTILES & IQR ─────────────────────────────────────────────────────────
q25, q50, q75 = np.percentile(scores, [25, 50, 75])
iqr = q75 - q25
print(f'\nQ1 (25%)  : {q25:.1f}')
print(f'Q2 (50%)  : {q50:.1f}   (= Median)')
print(f'Q3 (75%)  : {q75:.1f}')
print(f'IQR       : {iqr:.1f}   (Q3 - Q1 = spread of middle 50% of data)')
print(f'Outlier bounds: [{q25 - 1.5*iqr:.1f}, {q75 + 1.5*iqr:.1f}]  (1.5 × IQR rule)')

# ── VISUALISATION ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Basic Statistics', fontsize=13, fontweight='bold')

# Histogram
axes[0].hist(scores, bins=10, color='steelblue', edgecolor='black', alpha=0.8)
axes[0].axvline(mean,   color='red',    lw=2, linestyle='--', label=f'Mean={mean:.1f}')
axes[0].axvline(median, color='orange', lw=2, linestyle='--', label=f'Median={median:.1f}')
axes[0].axvspan(mean-std, mean+std, alpha=0.15, color='green', label=f'Mean ± Std')
axes[0].set_title('Score Distribution'); axes[0].set_xlabel('Score'); axes[0].set_ylabel('Count')
axes[0].legend()

# Box plot — shows median, IQR, and outliers visually
axes[1].boxplot(scores, vert=False, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].axvline(mean, color='red', lw=2, linestyle='--', label=f'Mean={mean:.1f}')
axes[1].set_title('Box Plot\n(box=IQR, line=median, whiskers=outlier bounds)')
axes[1].set_xlabel('Score'); axes[1].legend()

plt.tight_layout()
plt.show()

## 2 — Probability

ML models don't output certainties — they output **probabilities**.

| Concept | Definition | ML Example |
|---|---|---|
| **P(A)** | Probability of event A (0 to 1) | P(student passes) = 0.7 |
| **P(A\|B)** | Conditional: P(A given B happened) | P(pass \| studied ≥ 6 hours) |
| **P(A and B)** | Joint: both happen | P(high marks AND high attendance) |
| **P(A or B)** | Either happens | P(pass OR studied hard) |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
n = 1000

# Simulate a student dataset
study_hours = np.random.uniform(1, 10, n)
attendance  = np.random.uniform(40, 100, n)
noise       = np.random.normal(0, 8, n)
passed      = ((study_hours * 4 + attendance * 0.3 + noise) > 50).astype(int)
study_hard  = (study_hours >= 6).astype(int)   # studied 6+ hours = "studied hard"

# ── BASIC PROBABILITY ─────────────────────────────────────────────────────────
p_pass       = passed.mean()
p_hard       = study_hard.mean()
print(f'P(Pass)              = {p_pass:.3f}   ({int(p_pass*n)}/{n} students passed)')
print(f'P(Studied Hard)      = {p_hard:.3f}   ({int(p_hard*n)}/{n} studied 6+ hours)')

# ── JOINT PROBABILITY: P(A and B) ─────────────────────────────────────────────
p_hard_and_pass = (study_hard & passed).mean()
print(f'\nP(Hard AND Pass)     = {p_hard_and_pass:.3f}   (both conditions true)')

# ── CONDITIONAL PROBABILITY: P(Pass | Studied Hard) ──────────────────────────
# = P(Pass AND Hard) / P(Hard)
# "Given that a student studied hard, what is the probability they passed?"
p_pass_given_hard = p_hard_and_pass / p_hard
print(f'P(Pass | Hard)       = {p_pass_given_hard:.3f}   (if studied hard → {p_pass_given_hard*100:.0f}% chance of passing)')

# Compare to passing WITHOUT studying hard
p_not_hard     = 1 - p_hard
p_pass_no_hard = ((~study_hard.astype(bool)) & passed.astype(bool)).mean() / p_not_hard
print(f'P(Pass | NOT Hard)   = {p_pass_no_hard:.3f}   (if did NOT study hard → {p_pass_no_hard*100:.0f}% chance)')

print(f'\nConclusion: studying hard increases pass probability by {(p_pass_given_hard - p_pass_no_hard)*100:.0f} percentage points')

# ── PROBABILITY DISTRIBUTIONS ────────────────────────────────────────────────
print('\n=== Normal (Gaussian) Distribution ===')
# ML models assume features follow a normal distribution (bell curve)
# Defined by: mean (μ) and standard deviation (σ)
mu, sigma = 70, 15  # typical exam score distribution
samples = np.random.normal(mu, sigma, 10000)
samples = np.clip(samples, 0, 100)  # clamp to valid score range

print(f'μ (mean)  = {mu}  |  σ (std) = {sigma}')
print(f'68% of scores fall within [{mu-sigma}, {mu+sigma}]  (μ ± 1σ)')
print(f'95% of scores fall within [{mu-2*sigma}, {mu+2*sigma}] (μ ± 2σ)')
print(f'99% of scores fall within [{mu-3*sigma}, {mu+3*sigma}] (μ ± 3σ)')

# Logistic function (sigmoid) — used inside Logistic Regression
print('\n=== Sigmoid Function → converts any number to probability (0–1) ===')
z_values = np.linspace(-6, 6, 100)
sigmoid  = 1 / (1 + np.exp(-z_values))  # σ(z) = 1 / (1 + e^−z)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Probability in ML', fontsize=13, fontweight='bold')

axes[0].hist(samples, bins=50, color='steelblue', edgecolor='black', alpha=0.7, density=True)
axes[0].axvline(mu, color='red', lw=2, label=f'μ={mu}')
axes[0].axvspan(mu-sigma, mu+sigma, alpha=0.2, color='green', label='μ ± σ (68%)')
axes[0].set_title('Normal Distribution (exam scores)'); axes[0].set_xlabel('Score')
axes[0].set_ylabel('Density'); axes[0].legend()

axes[1].plot(z_values, sigmoid, color='steelblue', lw=2.5)
axes[1].axhline(0.5, color='red', linestyle='--', lw=1.5, label='Decision boundary (0.5)')
axes[1].axvline(0,   color='gray', linestyle='--', lw=1)
axes[1].fill_between(z_values, sigmoid, 0.5, where=sigmoid>0.5, alpha=0.15, color='green', label='Predict Positive')
axes[1].fill_between(z_values, sigmoid, 0.5, where=sigmoid<0.5, alpha=0.15, color='red',   label='Predict Negative')
axes[1].set_title('Sigmoid Function\nσ(z) = 1 / (1 + e⁻ᶻ)  ← core of Logistic Regression')
axes[1].set_xlabel('z (weighted sum of features)'); axes[1].set_ylabel('P(class = 1)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3 — Linear Algebra

In ML, every **sample is a vector** (a row of feature values) and every **dataset is a matrix** (a 2D grid of samples × features).

| Concept | Shape | ML use |
|---|---|---|
| **Vector** | (n,) or (n,1) | One sample's features, or model weights |
| **Matrix** | (rows, cols) | Whole dataset X: (n_samples, n_features) |
| **Dot Product** | scalar | Weighted sum of features — core of every linear model |
| **Matrix Multiply** | (m,n)×(n,k)→(m,k) | Predict for all samples at once |

In [ ]:
import numpy as np

# ── VECTORS ───────────────────────────────────────────────────────────────────
# A vector = an ordered list of numbers
# In ML: one student's features form a vector
student = np.array([6.0, 85.0, 72.0])    # [StudyHours, Attendance, PastScore]
weights = np.array([0.5, 0.3, 0.2])      # model's learned weights per feature

print('=== Vectors ===')
print(f'Student features : {student}   shape={student.shape}')
print(f'Model weights    : {weights}   shape={weights.shape}')

# Vector addition
print(f'\nVector addition: [1,2,3] + [4,5,6] = {np.array([1,2,3]) + np.array([4,5,6])}')

# Scalar multiplication
print(f'Scalar multiply : [1,2,3] × 2     = {np.array([1,2,3]) * 2}')

# Vector magnitude (length)
magnitude = np.linalg.norm(student)
print(f'Magnitude ||student|| = {magnitude:.2f}  (Euclidean length of the vector)')

# ── DOT PRODUCT — the core operation of every linear model ────────────────────
# dot(features, weights) = w1*x1 + w2*x2 + ... + wn*xn
# This is exactly what Linear Regression and Logistic Regression compute internally
dot_result = np.dot(student, weights)
manual     = student[0]*weights[0] + student[1]*weights[1] + student[2]*weights[2]
print(f'\n=== Dot Product (weighted sum) ===')
print(f'student · weights = {dot_result:.2f}')
print(f'Manual check      = {student[0]}×{weights[0]} + {student[1]}×{weights[1]} + {student[2]}×{weights[2]} = {manual:.2f}')
print(f'→ This "score" is passed through sigmoid → becomes P(Pass)')

# ── MATRICES ──────────────────────────────────────────────────────────────────
# A matrix = multiple vectors stacked as rows
# In ML: the whole dataset X is a matrix of shape (n_samples, n_features)
X = np.array([
    [6.0, 85.0, 72.0],   # Student 1: Study=6, Attendance=85, Past=72
    [3.0, 55.0, 45.0],   # Student 2
    [8.0, 92.0, 88.0],   # Student 3
    [2.0, 48.0, 38.0],   # Student 4
    [7.0, 88.0, 80.0],   # Student 5
])

print(f'\n=== Matrix X (dataset) ===')
print(X)
print(f'Shape: {X.shape}   → ({X.shape[0]} samples, {X.shape[1]} features)')

# Transpose: flip rows and columns
print(f'\nTranspose X.T shape: {X.T.shape}   → ({X.T.shape[0]} features, {X.T.shape[1]} samples)')

# ── MATRIX × VECTOR = predict all samples at once ────────────────────────────
# Instead of computing dot(student, weights) one at a time,
# X @ weights computes predictions for ALL students simultaneously
predictions = X @ weights   # @ is Python's matrix multiply operator
print(f'\n=== Matrix Multiply: X @ weights ===')
print(f'Weights: {weights}')
for i, (row, pred) in enumerate(zip(X, predictions)):
    print(f'  Student {i+1}: {row} → score={pred:.2f}')
print(f'→ All {X.shape[0]} predictions computed in ONE operation (no loop needed)')

# ── IDENTITY & INVERSE ────────────────────────────────────────────────────────
A = np.array([[2, 1], [5, 3]])
A_inv = np.linalg.inv(A)   # A × A_inv = Identity matrix
I = np.round(A @ A_inv)    # should be [[1,0],[0,1]]
print(f'\n=== Matrix Inverse (used in Normal Equation for Linear Regression) ===')
print(f'A =\n{A}')
print(f'A_inv =\n{A_inv.round(1)}')
print(f'A @ A_inv = Identity:\n{I.astype(int)}')

## 4 — Derivatives & Gradient Descent

This is **how models learn** — by repeatedly moving their parameters in the direction that reduces error.

**Key idea**: 
- A derivative tells you the *slope* (rate of change) at a point on a curve
- Gradient descent: start at a random point on the error curve → measure slope → step downhill → repeat until you reach the minimum

```
w = w - α × dL/dw
```
- `w` = model weight (parameter)
- `α` = learning rate (step size)
- `dL/dw` = gradient of loss with respect to w (slope)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── DERIVATIVE ───────────────────────────────────────────────────────────────
# Derivative of f(x) = slope of f at point x
# f'(x) = rate of change — positive slope = increasing, negative = decreasing

x = np.linspace(-3, 3, 200)

# Simple loss function: L(w) = w² (a parabola — minimum at w=0)
# f'(w) = 2w   → slope at any point = 2 × that point's value
L  = x**2
dL = 2 * x  # derivative

print('=== Understanding Derivatives ===')
for val in [-2, -1, 0, 1, 2]:
    slope = 2 * val
    direction = 'going up ↑' if slope > 0 else 'going down ↓' if slope < 0 else 'flat (minimum!) ─'
    print(f'  At w={val:+d}: slope = {slope:+d}  → {direction}')

# ── GRADIENT DESCENT SIMULATION ───────────────────────────────────────────────
print('\n=== Gradient Descent Simulation ===')
print('Loss function: L(w) = w²   |   Derivative: dL/dw = 2w')
print('Minimum is at w=0 (we know this — gradient descent will find it automatically)')

learning_rate = 0.3   # α — step size (too large = overshoot, too small = too slow)
w = 2.5               # starting point (random initialisation)
history = [w]

for step in range(10):
    gradient = 2 * w          # dL/dw at current w
    w = w - learning_rate * gradient   # move opposite to gradient (downhill)
    history.append(w)
    print(f'  Step {step+1:2d}: w={w:+.4f}  gradient={gradient:+.4f}  loss={w**2:.4f}')

print(f'\nFinal w = {w:.6f}  (true minimum is 0.0)')

# ── VISUALISATION ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Derivatives & Gradient Descent', fontsize=13, fontweight='bold')

# Plot 1: Loss curve with gradient descent path
axes[0].plot(x, L, 'steelblue', lw=2, label='L(w) = w²')
w_hist = np.array(history)
l_hist = w_hist**2
axes[0].plot(w_hist, l_hist, 'ro-', lw=2, ms=6, label='Gradient descent steps')
for i, (wi, li) in enumerate(zip(w_hist, l_hist)):
    axes[0].annotate(f'Step {i}', (wi, li), textcoords='offset points',
                     xytext=(5, 5), fontsize=7)
axes[0].set_xlabel('w (model weight)'); axes[0].set_ylabel('L(w) (loss / error)')
axes[0].set_title('Gradient Descent on L(w) = w²\n(rolling downhill to minimum)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Plot 2: Effect of learning rate
for lr, color, label in [(0.1, 'green', 'α=0.1 (slow)'),
                          (0.3, 'steelblue', 'α=0.3 (good)'),
                          (0.9, 'red', 'α=0.9 (oscillates)')]:
    w_lr = 2.5
    hist_lr = [w_lr]
    for _ in range(15):
        w_lr = w_lr - lr * 2 * w_lr
        hist_lr.append(w_lr)
    axes[1].plot(range(len(hist_lr)), [v**2 for v in hist_lr], 'o-',
                 color=color, lw=1.5, ms=4, label=label)
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Loss L(w)')
axes[1].set_title('Effect of Learning Rate α\n(too small=slow, too large=unstable)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('\nKey insight: scikit-learn models run gradient descent automatically inside .fit()')

## 5 — Distance Metrics

Algorithms like **KNN** and **K-Means** classify or cluster data by measuring "closeness" between points.

| Metric | Formula | When to use |
|---|---|---|
| **Euclidean** | √Σ(a−b)² | Default — straight-line distance in space |
| **Manhattan** | Σ\|a−b\| | City-block distance — robust to outliers |
| **Cosine** | 1 − (a·b)/(‖a‖‖b‖) | Angle between vectors — used in text similarity |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Two students represented as feature vectors [StudyHours, Attendance]
student_A = np.array([6.0, 85.0])   # [hours, attendance]
student_B = np.array([3.0, 55.0])
student_C = np.array([6.5, 88.0])   # close to A

# ── EUCLIDEAN DISTANCE ─────────────────────────────────────────────────────────
# = √( (a1-b1)² + (a2-b2)² + ... )
# Straight-line distance in feature space
def euclidean(a, b):
    return np.sqrt(np.sum((a - b)**2))

print('=== Euclidean Distance (default in KNN, K-Means) ===')
print(f'd(A, B) = √(({student_A[0]}-{student_B[0]})² + ({student_A[1]}-{student_B[1]})²)')
print(f'       = √({(student_A[0]-student_B[0])**2:.0f} + {(student_A[1]-student_B[1])**2:.0f})')
print(f'       = {euclidean(student_A, student_B):.2f}')
print(f'd(A, C) = {euclidean(student_A, student_C):.2f}  ← C is closer to A than B is')

# ── MANHATTAN DISTANCE ────────────────────────────────────────────────────────
# = |a1-b1| + |a2-b2| + ...
# Sum of absolute differences — like walking on a grid (city blocks)
def manhattan(a, b):
    return np.sum(np.abs(a - b))

print(f'\n=== Manhattan Distance (city-block) ===')
print(f'd_manhattan(A, B) = |{student_A[0]}-{student_B[0]}| + |{student_A[1]}-{student_B[1]}|')
print(f'                 = {abs(student_A[0]-student_B[0]):.0f} + {abs(student_A[1]-student_B[1]):.0f} = {manhattan(student_A, student_B):.2f}')

# ── WHY SCALING MATTERS FOR DISTANCE ────────────────────────────────────────
print('\n=== Why Feature Scaling is Critical for Distance-Based Algorithms ===')
# Without scaling, Attendance (40–100 range) dominates StudyHours (1–10 range
# A 1-hour difference in study = 1 unit
# A 1% difference in attendance = 1 unit  ← same raw difference but different meaning!
diff_hours      = abs(student_A[0] - student_B[0])   # 3 hours
diff_attendance = abs(student_A[1] - student_B[1])   # 30 points
print(f'Unscaled: diff_hours={diff_hours:.0f}, diff_attendance={diff_attendance:.0f}')
print(f'  → Attendance dominates ({diff_attendance/(diff_hours+diff_attendance)*100:.0f}% of total distance)')
print(f'  → StudyHours is effectively ignored!')
print(f'  → Fix: use StandardScaler before computing distances')

# ── VISUALISATION ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Distance Metrics — Why They Matter in KNN', fontsize=13, fontweight='bold')

# Plot 1: Points and distances
points = {'A': student_A, 'B': student_B, 'C': student_C}
colors = {'A': 'steelblue', 'B': 'tomato', 'C': 'mediumseagreen'}
for name, pt in points.items():
    axes[0].scatter(*pt, s=150, color=colors[name], zorder=5, label=f'Student {name}: {pt}')
    axes[0].annotate(name, pt, textcoords='offset points', xytext=(8, 5), fontsize=12, fontweight='bold')
# Draw lines for distance
axes[0].plot([student_A[0], student_B[0]], [student_A[1], student_B[1]], 'k--', lw=1.5,
             label=f'A–B Euclidean={euclidean(student_A,student_B):.1f}')
axes[0].plot([student_A[0], student_C[0]], [student_A[1], student_C[1]], 'g--', lw=1.5,
             label=f'A–C Euclidean={euclidean(student_A,student_C):.1f}')
axes[0].set_xlabel('Study Hours'); axes[0].set_ylabel('Attendance %')
axes[0].set_title('Feature Space — KNN finds K nearest points'); axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Plot 2: Euclidean vs Manhattan visually
a, b = np.array([1, 1]), np.array([4, 4])
axes[1].scatter(*a, s=200, color='steelblue', zorder=5, label='Point A')
axes[1].scatter(*b, s=200, color='tomato',    zorder=5, label='Point B')
# Euclidean: straight line
axes[1].plot([a[0], b[0]], [a[1], b[1]], 'b-', lw=2.5, label=f'Euclidean = {euclidean(a,b):.2f}')
# Manhattan: L-shaped path
axes[1].plot([a[0], b[0], b[0]], [a[1], a[1], b[1]], 'r--', lw=2.5,
             label=f'Manhattan = {manhattan(a,b):.2f}')
axes[1].set_xlim(0, 5); axes[1].set_ylim(0, 5); axes[1].grid(True)
axes[1].set_title('Euclidean (straight) vs Manhattan (city-block)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6 — Error & Cost Functions

A **cost function** (also called loss function) measures how wrong the model's predictions are.
Gradient descent minimises this value during training.

| Loss Function | Used for | Formula |
|---|---|---|
| **MSE** (Mean Squared Error) | Regression | (1/n) Σ(y − ŷ)² |
| **Log Loss** (Binary Cross-Entropy) | Classification | −(y·log(ŷ) + (1−y)·log(1−ŷ)) |

**Intuition**: higher loss = model is more wrong → gradient descent pushes parameters to reduce it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── MSE COST FUNCTION ─────────────────────────────────────────────────────────
# Used in Linear Regression
# MSE = (1/n) × Σ (actual - predicted)²
# Squaring: (1) makes all errors positive, (2) punishes large errors more

actual    = np.array([70.0, 85.0, 55.0, 90.0, 60.0])
predicted = np.array([72.0, 80.0, 60.0, 88.0, 65.0])

errors   = actual - predicted               # residuals: positive=under-predicted, negative=over
sq_errors = errors**2                       # squared residuals
mse      = sq_errors.mean()                  # average of squared errors
rmse     = np.sqrt(mse)                     # back to original units

print('=== MSE Cost Function (Regression) ===')
print(f'{"Sample":<8} {"Actual":<8} {"Predicted":<12} {"Error":<8} {"Error²"}')
for i, (a, p, e, se) in enumerate(zip(actual, predicted, errors, sq_errors), 1):
    print(f'{i:<8} {a:<8.0f} {p:<12.0f} {e:<8.0f} {se:.0f}')
print(f'{"":-<50}')
print(f'{"":<8} {"":<8} {"":<12} {"":<8} {sq_errors.sum():.0f}  ← sum of squared errors')
print(f'MSE  = {sq_errors.sum():.0f} / {len(actual)} = {mse:.2f}')
print(f'RMSE = √{mse:.2f} = {rmse:.2f}  (in same units as scores)')

# ── LOG LOSS (Binary Cross-Entropy) ──────────────────────────────────────────
# Used in Logistic Regression (classification)
# Penalises CONFIDENT wrong predictions very harshly
# If actual=1 (Pass) and model says P(Pass)=0.01 → huge loss (model was very wrong)
# If actual=1 (Pass) and model says P(Pass)=0.99 → tiny loss (model was confident+correct)

print('\n=== Log Loss (Classification) ===')
print('Log loss penalises confident wrong predictions very harshly\n')

actual_classes = np.array([1, 0, 1, 1, 0])          # 1=Pass, 0=Fail
prob_pass      = np.array([0.9, 0.1, 0.8, 0.3, 0.7])  # model's P(Pass)

print(f'{"Actual":<8} {"P(Pass)":<12} {"Log Loss":<12} {"Note"}')
for a, p in zip(actual_classes, prob_pass):
    p_clip = np.clip(p, 1e-7, 1 - 1e-7)  # avoid log(0) = -infinity
    loss = -(a * np.log(p_clip) + (1-a) * np.log(1-p_clip))
    outcome = 'Correct' if (a==1 and p>0.5) or (a==0 and p<=0.5) else 'WRONG'
    print(f'{a:<8} {p:<12.2f} {loss:<12.4f} {outcome}')

# Full log loss
eps = 1e-7
full_log_loss = -np.mean(
    actual_classes * np.log(prob_pass + eps) +
    (1-actual_classes) * np.log(1-prob_pass + eps)
)
print(f'\nOverall Log Loss = {full_log_loss:.4f}')
print('Interpretation: lower = better | 0 = perfect | ~0.69 = random guessing')

# ── VISUALISATION ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Error & Cost Functions', fontsize=13, fontweight='bold')

# Plot 1: MSE — show how model improves as weight changes
w_range = np.linspace(-3, 5, 200)
# Toy example: X=2, y_true=4, prediction = w * X
y_true_toy = 4.0
X_toy      = 2.0
mse_vals   = (y_true_toy - w_range * X_toy)**2  # MSE for each w
best_w     = y_true_toy / X_toy  # optimal w = 2.0

axes[0].plot(w_range, mse_vals, 'steelblue', lw=2)
axes[0].axvline(best_w, color='red', lw=2, linestyle='--', label=f'Optimal w={best_w:.1f} (MSE=0)')
axes[0].set_xlabel('Model Weight w'); axes[0].set_ylabel('MSE Loss')
axes[0].set_title('MSE Loss Curve\n(gradient descent finds the minimum)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Plot 2: Log loss — shows confident wrong predictions are punished harshly
p_values = np.linspace(0.001, 0.999, 300)
loss_y1  = -np.log(p_values)    # log loss when actual=1
loss_y0  = -np.log(1-p_values)  # log loss when actual=0

axes[1].plot(p_values, loss_y1, 'steelblue', lw=2, label='Actual = 1 (Pass)')
axes[1].plot(p_values, loss_y0, 'tomato',    lw=2, label='Actual = 0 (Fail)')
axes[1].axvline(0.5, color='gray', linestyle='--', lw=1.5, label='Decision boundary')
axes[1].set_xlim(0, 1); axes[1].set_ylim(0, 5)
axes[1].set_xlabel('P(Pass) — model output'); axes[1].set_ylabel('Log Loss')
axes[1].set_title('Log Loss\n(confident wrong predictions = very high loss)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Maths Summary — How It All Connects to ML

```
Statistics  →  understand your data BEFORE modelling
               mean, median, std tell you the shape of your features
               outlier check (IQR) → clean data = better models

Probability →  models output probabilities, not certainties
               P(Pass | features) = sigmoid(w · x)
               ROC-AUC, precision, recall are all built on probability

Linear Algebra → your dataset IS a matrix   X.shape = (n_samples, n_features)
               prediction for all samples = X @ weights  (one operation)
               dot product = weighted sum of features = core of every linear model

Derivatives  →  how .fit() works internally
               gradient = slope of the loss curve at current weights
               gradient descent: w = w - α × gradient   (repeat until minimum)

Distance     →  KNN predicts by finding the K nearest training points
               K-Means clusters by assigning each point to nearest centroid
               ALWAYS scale features before using distance-based algorithms

Cost Function → what the model minimises during training
               Regression: MSE = (1/n) Σ(y - ŷ)²
               Classification: Log Loss = -Σ[y·log(ŷ) + (1-y)·log(1-ŷ)]
               Lower loss = better model
```